In [0]:
# ============================================================
# Silver — Source 18: AWS SES Email Events
#
# Transformations:
#   - Cast sent_at, delivered_at, bounced_at, complained_at to timestamp
#   - Normalise event_type, delivery_status
#   - customer_id null is valid (some email types)
#   - Reject null message_id or order_id → quarantine
#   - Deduplicate on message_id
#
# Source:  bronze.src_18_email.email_events
# Target:  silver.src_18_email.email_events
# Quarantine: silver.quarantine.src_18_email
# ============================================================
from pyspark.sql import functions as F
from delta.tables import DeltaTable
BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_18_email.email_events'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_18_email'
VALID_EVENT_TYPES = ['order_confirmation', 'shipping_notification', 'delivery_confirmation',
                     'review_request', 'return_confirmation', 'promotional', 'password_reset',
                     'refund_confirmation', 'marketing_weekly', 'marketing_promo']
VALID_STATUSES = ['delivered', 'bounced', 'complained', 'pending', 'failed', 'sent']
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_18_email')
print('Silver Source 18 SES Email — starting...')


In [0]:
bronze = spark.table(f'{BRONZE_CATALOG}.src_18_email.email_events')
total = bronze.count()
print(f'Bronze rows: {total}')

# Cast timestamps
df = bronze \
    .withColumn('sent_at',       F.to_timestamp(F.col('sent_at'))) \
    .withColumn('delivered_at',  F.to_timestamp(F.col('delivered_at'))) \
    .withColumn('bounced_at',    F.to_timestamp(F.col('bounced_at'))) \
    .withColumn('complained_at', F.to_timestamp(F.col('complained_at')))

# Normalise
df = df \
    .withColumn('event_type',      F.lower(F.trim(F.col('event_type')))) \
    .withColumn('delivery_status', F.lower(F.trim(F.col('delivery_status'))))

# Bad rows — message_id and order_id required
# customer_id null is valid (review_request type)
bad = df.filter(
    F.col('message_id').isNull() |
    F.col('order_id').isNull() |
    F.col('sent_at').isNull() |
    ~F.col('event_type').isin(VALID_EVENT_TYPES) |
    ~F.col('delivery_status').isin(VALID_STATUSES)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('email_events'))

good = df.filter(
    F.col('message_id').isNotNull() &
    F.col('order_id').isNotNull() &
    F.col('sent_at').isNotNull() &
    F.col('event_type').isin(VALID_EVENT_TYPES) &
    F.col('delivery_status').isin(VALID_STATUSES)
).dropDuplicates(['message_id'])

bad_count = bad.count()
good_count = good.count()
print(f'Email events: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')

# Show delivery stats
df.groupBy('delivery_status').count().orderBy('count', ascending=False).show()

if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.message_id = s.message_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
print('Written')

if bad_count > 0:
    bad.select(
        F.lit('src_18_email').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    ).write.format('delta').mode('append').option('mergeSchema','true').saveAsTable(QUARANTINE_TABLE)
    print(f'{bad_count} quarantined')


In [0]:
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_18_email.email_events: {count} rows')
spark.sql(f'''
    SELECT event_type, delivery_status, COUNT(*) as cnt
    FROM {TARGET_TABLE}
    GROUP BY event_type, delivery_status
    ORDER BY cnt DESC
''').show()
